# Paper Reproduction Template

Companion notebook for the [Reading and Reproducing Papers](https://codemia.ai/courses/applied_ai_systems/reading_and_reproducing_papers) lesson on Codemia.

**What you will build**: a working reproduction of a small, well-defined empirical claim from a recent ML paper — specifically the "lost-in-the-middle" finding from Liu et al. (2023) — plus a reusable template structure you can adapt to reproduce any paper.

**Why this paper**: it has a self-contained, low-compute experiment (measure retrieval accuracy as a function of where a fact is placed in the context window) that can be run against any instruction-tuned model in a few minutes. The claim is sharp: LLMs are much better at using information placed at the beginning or end of the context than in the middle. Perfect for a teaching reproduction.

**Reference**:
- Liu et al., [Lost in the Middle: How Language Models Use Long Contexts](https://arxiv.org/abs/2307.03172) (TACL 2024)

**Time**: ~45 minutes  **GPU required**: yes (T4 or better)

## How to use this notebook as a template

Every paper reproduction has the same five phases. Each phase is one or two sections of this notebook:

1. **Read the paper and extract the claim** — what exactly is being measured, and what's the concrete prediction you could disprove?
2. **Set up the baseline** — find or load the dataset, pick a model, write the evaluation harness.
3. **Implement the technique** — for this paper, it's the "reorder documents by gold position" trick. For yours, it could be a new architecture, a new loss, a new decoding strategy.
4. **Run the comparison** — baseline vs technique, plot the result, check if the curves match the paper's.
5. **Write the reproduction report** — what matched, what didn't, what you learned.

The scaffolding below is the structure. Rip it out, put your paper's specifics in.

## Setup

In [ ]:
%pip install -q transformers accelerate datasets torch matplotlib

In [ ]:
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Phase 1: Extract the claim

Quoting the paper abstract:

> We analyze the performance of language models on two tasks that require identifying relevant information in their input contexts: multi-document question answering and key-value retrieval. **We find that performance is often highest when relevant information occurs at the beginning or end of the input context, and significantly degrades when models must access relevant information in the middle of long contexts.**

Concrete prediction we can test: if we run a key-value retrieval task where the key we query is placed at position `k` out of `N` distractor keys, accuracy should form a **U-shape** as a function of `k`. High at `k=0` (start), high at `k=N-1` (end), lowest around `k=N/2` (middle).

**What makes this reproducible**:
- The task is synthetic (we generate the data ourselves — no dataset licensing).
- It is zero-shot (no training).
- It works on any autoregressive LM that takes ~4k tokens of context.
- The whole experiment is maybe 200 forward passes.

**What to write down before coding**:
- Hypothesis: accuracy vs. key position is U-shaped.
- Null result: accuracy is flat (no middle degradation).
- Disconfirming result: accuracy monotonically increases or decreases.

## Phase 2: Set up the baseline

Pick a small instruction-tuned model that fits on T4. TinyLlama-1.1B-Chat is the smallest widely-used one. Phi-2 or Gemma-2B are other good choices. For the official reproduction you would want a "real" LLM (GPT-3.5, Llama-2-7B-chat), but the U-shape is observable even in smaller models.

In [ ]:
MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
# If the TinyLlama download fails, 'microsoft/phi-2' (~2.7B) also works on T4 with fp16.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto'
)
model.eval()
print('model loaded:', sum(p.numel() for p in model.parameters())/1e6, 'M params')

### Build the key-value retrieval task

We generate synthetic key-value pairs, each with a random UUID-like key and a random short value. The prompt asks "what is the value for key X?" and we check whether the exact value string appears in the model's output.

In [ ]:
def random_key():
    # 6 random lowercase letters. Short enough to stay tokenized as ~2-3 tokens,
    # unique enough that collisions are negligible for N<100.
    return ''.join(random.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(6))

def random_value():
    # 6-digit integer as a string. Easy to check for exact match.
    return str(random.randint(100_000, 999_999))

def make_kv_prompt(n_pairs, gold_position):
    # Generate n_pairs keys, decide which one is the gold key, and build the prompt.
    keys = [random_key() for _ in range(n_pairs)]
    values = [random_value() for _ in range(n_pairs)]
    gold_key = keys[gold_position]
    gold_value = values[gold_position]
    lines = [f'{k}: {v}' for k, v in zip(keys, values)]
    context = '\n'.join(lines)
    prompt = (
        'Here is a list of key-value pairs:\n'
        f'{context}\n\n'
        f'What is the value associated with the key "{gold_key}"? '
        'Answer with only the value.'
    )
    return prompt, gold_value

p, v = make_kv_prompt(n_pairs=5, gold_position=2)
print(p)
print('gold value:', v)

In [ ]:
def generate(prompt, max_new_tokens=20):
    # Greedy decoding so the measurement is deterministic.
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Strip off the prompt tokens to get just the generated completion.
    return tokenizer.decode(out[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

def is_correct(output, gold_value):
    # Loose match: as long as the 6-digit string appears anywhere in the output,
    # count it as correct. Avoids punishing the model for verbosity.
    return gold_value in output

# Sanity check on a 5-pair problem.
prompt, gold = make_kv_prompt(n_pairs=5, gold_position=2)
out = generate(prompt)
print(f'model said: {out!r}')
print(f'gold: {gold}  correct: {is_correct(out, gold)}')

## Phase 3: Run the main experiment

For each gold position from 0 to N-1, generate `n_trials` random KV sets with the key at that position, query the model, measure accuracy. Plot accuracy vs. position.

In [ ]:
def run_experiment(n_pairs=20, n_trials=10):
    results = []
    for pos in range(n_pairs):
        correct = 0
        for _ in range(n_trials):
            prompt, gold = make_kv_prompt(n_pairs=n_pairs, gold_position=pos)
            out = generate(prompt)
            if is_correct(out, gold):
                correct += 1
        acc = correct / n_trials
        results.append(acc)
        print(f'position {pos:2d}/{n_pairs-1}: {acc:.2f}')
    return results

# 20 pairs = ~400 tokens context — modest but enough to see the effect on a 1B model.
# Bigger models can handle 75+ pairs and show the effect more dramatically.
accs = run_experiment(n_pairs=20, n_trials=8)

In [ ]:
positions = list(range(len(accs)))
plt.figure(figsize=(7, 4))
plt.plot(positions, accs, 'o-', linewidth=2)
plt.xlabel('Gold key position (0=start, 19=end)')
plt.ylabel('Retrieval accuracy')
plt.title(f'"Lost in the Middle" reproduction on {MODEL_ID.split("/")[-1]}')
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Phase 4: Interpret the results

Look at the plot. Did you see a U-shape? Answer these questions honestly:

- **Match the paper's shape?** The paper reports a clear U for GPT-3.5 and Claude. On a small model like TinyLlama-1.1B you may see a noisier version, or a curve that's mostly high-at-start and degrades everywhere else.
- **Effect size match?** Paper says ~20 percentage points between best and worst positions. How much do you see? (Be honest — this is a smaller model and smaller context, so expect effects to be weaker or noisier.)
- **Noise floor**: with only 8 trials per position, the standard error on each point is ~0.17. That's big. Real reproduction runs use 100+ trials per condition.

The point of reproduction is not to reproduce numbers exactly — no one ever does — but to reproduce **the qualitative phenomenon**. If you see the U-shape at all, the paper's core claim holds on your setup. If you don't, that is *also* a useful finding and deserves a bullet in your report.

## Phase 5: Write the reproduction report

Use the template below. Fill in the blanks with what you actually found. A good reproduction report is ~300 words and has four sections.

In [ ]:
report_template = '''
# Reproduction Report: Lost in the Middle (Liu et al. 2023)

## Setup
- Model: {model}
- Task: synthetic key-value retrieval with {n_pairs} pairs per prompt
- Trials per position: {n_trials}
- Date: fill in

## Result
- Accuracy at position 0 (start): {acc_start:.2f}
- Accuracy at position {mid_pos} (middle): {acc_mid:.2f}
- Accuracy at position {last_pos} (end): {acc_end:.2f}
- Minimum accuracy: {acc_min:.2f} (at position {min_pos})
- Maximum accuracy: {acc_max:.2f} (at position {max_pos})
- U-shape detected: {u_shape}

## Discussion
- The qualitative finding of the paper {holds_or_not}.
- Differences from the paper's exact numbers: our model is {model_size}x smaller
  than GPT-3.5 and we used a single-context, fixed-length task. The effect is
  expected to be weaker on small models.
- Noise: with only {n_trials} trials per position, the standard error on each
  point is roughly {noise:.2f}. That would need {recommended_trials}+ trials to
  tighten to 2 percentage points.

## Things to fix or extend
- Run against a larger model (Llama-3-8B-instruct) to see if the effect sharpens.
- Vary the number of pairs. The paper shows the effect growing with longer contexts.
- Use the paper's exact prompt format (theirs includes specific few-shot examples).
'''


In [ ]:
mid_pos = len(accs) // 2
last_pos = len(accs) - 1
acc_min = min(accs)
acc_max = max(accs)
# Simple U-shape detector: min accuracy should be in the middle third.
middle_third = set(range(len(accs)//3, 2*len(accs)//3))
u_shape = 'YES' if accs.index(acc_min) in middle_third else 'NO / noisy'

report = report_template.format(
    model=MODEL_ID,
    n_pairs=20,
    n_trials=8,
    acc_start=accs[0],
    acc_mid=accs[mid_pos],
    acc_end=accs[last_pos],
    mid_pos=mid_pos,
    last_pos=last_pos,
    acc_min=acc_min,
    acc_max=acc_max,
    min_pos=accs.index(acc_min),
    max_pos=accs.index(acc_max),
    u_shape=u_shape,
    holds_or_not='holds' if u_shape == 'YES' else 'is weakly supported',
    model_size='~150',
    noise=0.17,
    recommended_trials=100,
)
print(report)

## Variations (phase 3 extensions)

Try two things to stress the finding:

1. **Longer contexts**: does the U-shape sharpen as `n_pairs` grows?
2. **Bigger model**: swap in a 3B or 7B model and see if the effect gets sharper or flatter.

In [ ]:
# Variation 1: longer context. We expect the middle dip to deepen as n_pairs grows.
# WARNING: this runs ~150 forward passes; give it a minute.
n_pairs_small = run_experiment(n_pairs=10, n_trials=6)
n_pairs_big = run_experiment(n_pairs=30, n_trials=6)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.linspace(0, 1, 10), n_pairs_small, 'o-', label='10 pairs')
ax.plot(np.linspace(0, 1, 20), accs, 's-', label='20 pairs')
ax.plot(np.linspace(0, 1, 30), n_pairs_big, '^-', label='30 pairs')
ax.set_xlabel('Normalized gold position (0=start, 1=end)')
ax.set_ylabel('Retrieval accuracy')
ax.set_title('Context length vs. middle degradation')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## How to adapt this template to a different paper

Swap the content in each phase for your target paper:

| Phase | Generic version | Example for "Lost in the Middle" |
|-------|-----------------|----------------------------------|
| 1. Claim | The one-sentence prediction the paper makes | "Accuracy vs. gold position is U-shaped" |
| 2. Baseline | Whatever setup the paper compares against | TinyLlama + zero-shot prompt |
| 3. Technique | The new thing the paper introduces | N/A — no technique, just a measurement |
| 4. Comparison | Plot baseline vs. technique, match shape | Plot accuracy vs. position |
| 5. Report | 300 words: match, diff, limitations | See template above |

For a paper that introduces a method (e.g., a new attention variant, a new optimizer), phase 3 is where you implement the method and phase 4 is where you compare baseline vs. method on the paper's benchmark.

**Picking a reproducible paper**:
- Small, self-contained experiments. Avoid papers whose main result needs 1000 GPU-hours.
- Synthetic or publicly-available datasets. Avoid proprietary eval sets.
- Sharp qualitative claims. "U-shaped" is testable. "Improved performance" is not.
- Recent (<2 years). Older papers often depend on libraries that no longer work.
- Bonus: paper releases code. You can sanity-check your implementation against theirs.

## What to try next

- Reproduce a paper from your own reading queue. Use the template above — the scaffolding is already written.
- Run this exact experiment on GPT-4 via API. The effect is much sharper on larger models.
- Try the multi-document QA variant from the paper: 20 Wikipedia paragraphs, one of which contains the answer, placed at varying positions. More realistic, harder to build.
- Propose a mitigation. The paper suggests positional encoding as a candidate cause — can you confirm that by swapping the base model for one with a different positional scheme (ALiBi, RoPE, NoPE)?
- Write up your reproduction as a blog post. Genuine reproductions are rare and valuable.